# Notebook 065: Medallion Architecture: Silver Transformation, Cleansing and Conformance

**Module:** Module 07: Lakehouse Data Modeling, CDC & SCD Implementation  
**Difficulty:** `Intermediate`  
**Databricks Masterclass Reference:** Video 17: Lakeflow Declarative Pipelines & Video 20: Finance Domain E2E Project  

---

## 1. Business Scenario & Real-World Context
Raw invoices from the Bronze layer contain duplicate entries, unstandardized status codes ('appr', 'APPROVED', 'approved'), missing tax codes, and stringified numeric values. The Silver layer must conform and cleanse these into enterprise-ready records.

## 2. Technical Requirement & Problem Statement
Build the Silver layer transformation: filter and clean raw data, standardize categorical codes, enforce data quality rules (non-negative amounts), cast data to strict primitive types, and execute deterministic deduplication into a conformed Delta Silver table.

## 3. Spark & Databricks Architecture Notes
The Silver layer represents the enterprise-conformed, cleansed version of truth. It standardizes naming conventions, parses strings into structured types, deduplicates records using business keys, and joins disparate Bronze feeds into consolidated entity tables.


## 4. Environment Setup & SparkSession Initialization
We configure a resilient `SparkSession` compatible with local PySpark development, Databricks runtime, or CI/CD test runners.

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# Initialize SparkSession with Delta Lake support and optimized local configs
spark = (
    SparkSession.builder
    .appName(f"UseCase_065_Medallion_Architecture_Silver_Transformation,_Cleansing_and_Conformance")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.default.parallelism", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f" Spark Version: {spark.version}")
print(f" Session Active: {spark.sparkContext.appName}")


## 5. Reading from Upstream Bronze Layer
Read uncleaned records containing duplicates and messy categories.

In [ ]:
raw_bronze = [
    ("INV-1001", "V-01", "1,500.00", "appr", "2023-11-20 10:00:00"),
    ("INV-1001", "V-01", "1,500.00", "APPROVED", "2023-11-20 10:05:00"), # Duplicate update
    ("INV-1002", "V-02", "420.50", "PENDING", "2023-11-20 10:01:00"),
    ("INV-1003", "V-03", "-50.00", "REJECTED", "2023-11-20 10:02:00")   # Invalid negative invoice!
]

schema = "invoice_no STRING, vendor_id STRING, raw_amount STRING, raw_status STRING, updated_at STRING"
df_bronze_in = spark.createDataFrame(raw_bronze, schema)
df_bronze_in.show()

## 6. Applying Cleansing, Type Casting and Status Standardization
Standardize uppercase status, clean currency strings, and enforce positive amounts.

In [ ]:
# Clean amount and standardize status
df_cleansed = (
    df_bronze_in
    .withColumn("clean_amount", F.regexp_replace("raw_amount", r"[^\d.]", "").cast(T.DoubleType()))
    .withColumn(
        "standard_status",
        F.when(F.upper(F.col("raw_status")).isin(["APPR", "APPROVED"]), "APPROVED")
         .when(F.upper(F.col("raw_status")).isin(["PEND", "PENDING"]), "PENDING")
         .otherwise(F.upper(F.col("raw_status")))
    )
    # Quality filter: reject invoices with non-positive amounts
    .filter(F.col("clean_amount") > 0)
)

df_cleansed.show()

## 7. Deterministic Deduplication and Silver Table Landing
Retain only the latest update per invoice_no using window deduplication.

In [ ]:
silver_path = "/tmp/lakehouse_silver_065"
if Path(silver_path).exists():
    shutil.rmtree(silver_path)

win_dedup = Window.partitionBy("invoice_no").orderBy(F.col("updated_at").desc())

df_silver = (
    df_cleansed
    .withColumn("rn", F.row_number().over(win_dedup))
    .filter(F.col("rn") == 1)
    .select(
        "invoice_no",
        "vendor_id",
        F.col("clean_amount").alias("invoice_amount"),
        F.col("standard_status").alias("invoice_status"),
        F.to_timestamp("updated_at").alias("last_updated_at"),
        F.current_timestamp().alias("_silver_processed_at")
    )
)

df_silver.write.format("delta").mode("overwrite").save(silver_path)

print("Conformed Silver Delta Table:")
spark.read.format("delta").load(silver_path).show(truncate=False)

## 9. Key Architectural Takeaways
- **Silver layer provides conformed, cleansed, deduplicated data for cross-functional consumption.**
- **Standardize categorical codes and parse strings into strictly typed columns.**
- **Always filter out records violating fundamental business domain rules before landing in Silver.**

## 10. Senior Data Engineering Interview Questions
1. What transformations belong in the Silver layer versus the Gold layer?
2. How do you handle records that fail data quality rules during Silver transformation?
3. How do you ensure Silver tables maintain low query latency for analytical users?

---
*Created for the **Spark Coding Use Cases Masterclass** repository referencing the Databricks All-in-One Playlist.*
